## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 12 samples and 14441 columns.
First few rows of the data:


,os,os event,pfs,pfs event,sample_title,tissue,treatment,overall response,age,biomaterial provider,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,6.505133,1,2.004107,1,R009,breast,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Progressive Disease,49,"Alberto J. Montero,alberto.montero@uhhospitals...",...,7.886025,5.469030,7.033145,8.653241,4.809948,5.473786,7.548196,6.930863,6.056256,7.738258
1,41.330595,1,8.344969,1,R010_PRE,unspecified metastatic site,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Partial Response,62,"Alberto J. Montero,alberto.montero@uhhospitals...",...,6.810587,4.255260,5.839894,6.943901,3.219846,4.129328,6.085884,6.036035,4.764023,6.600902
2,11.039014,1,11.039014,0,R012,lymph node,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Partial Response,70,"Alberto J. Montero,alberto.montero@uhhospitals...",...,7.459401,5.454017,7.141900,7.864083,5.298725,5.802623,7.851647,7.127841,5.987570,8.080737
3,4.928131,1,4.928131,0,R013,lung,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Partial Response,45,"Alberto J. Montero,alberto.montero@uhhospitals...",...,7.354993,4.727607,6.573172,7.545722,4.605355,5.074363,7.311338,6.425795,5.911971,6.762733
4,18.759754,1,1.905544,1,R014_PRE,sternum,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Progressive Disease,36,"Alberto J. Montero,alberto.montero@uhhospitals...",...,7.486330,5.415407,7.048296,7.943261,4.677227,5.441739,7.530151,7.112542,6.053870,7.572538


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
